# RobotMath2030 — Sinkhorn Optimal Transport (Ch.05)

Soft correspondence for misaligned point clouds.

Full chapter: [05_sinkhorn_point_clouds](../chapters/05_sinkhorn_point_clouds/)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e .
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from miniworlds.point_cloud_world import misaligned_pair
from robotmath.optimal_transport import sinkhorn2, pairwise_sq_distances
from robotmath.viz import draw_point_clouds, draw_transport_plan

source, target = misaligned_pair(seed=7, theta=0.4, tx=0.2, ty=-0.1, noise=0.025)
rng = np.random.default_rng(11)
target = target[rng.permutation(target.shape[0])]

w2_cost, plan, gaps = sinkhorn2(source, target, epsilon=0.03, max_iter=300)
idx_cost = float(np.mean(np.sum((source[: target.shape[0]] - target[: source.shape[0]]) ** 2, axis=1)))

print(f'Sinkhorn W2^2 approx: {w2_cost:.5f}')
print(f'Naive index L2:       {idx_cost:.5f}')
print(f'Marginal gap (final): {gaps[-1]:.2e}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
draw_point_clouds(axes[0], source, target)
axes[0].set_title('Misaligned scan pair')
draw_transport_plan(axes[1], source, target, plan, threshold=0.0015, max_lines=50)
draw_point_clouds(axes[1], source, target)
axes[1].set_title('Sinkhorn transport plan')
plt.tight_layout()
plt.show()